<a href="https://colab.research.google.com/github/josephgalicinao/SkinLesionDetection/blob/main/Skin_Lesion_Classical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Initialization

## Imports

Important imports and packages used

In [ ]:
from google.colab import userdata
import os
import kagglehub
import pandas as pd
import tensorflow as tf
import os
import csv
import numpy as np
import matplotlib.pyplot as plt
import cv2
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from mlxtend.classifier import StackingClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, precision_recall_curve, auc

import csv
import pandas as pd
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

IMG_SIZE   = (128, 128)
AUTOTUNE   = tf.data.AUTOTUNE

## Load Dataset

In [ ]:
# Load the HAM10000 dataset
ham10000_dataset_path = kagglehub.dataset_download("josephgalicinao/skin-lesion-dataset")

print("Path to dataset files:", ham10000_dataset_path)

## Create Datasets

Load data and masks

In [ ]:
def get_HAM10000_paths(image_ids):
    '''
    Gets the paths of each image, mask, and distance map based on the image ID
    Makes sure that the image, mask, and distance map exist
    image_ids: list of image IDs
    returns: img_paths, mask_paths, distance_paths
    '''

    img_paths = []
    mask_paths = []
    distance_paths = []
    for img_id in image_ids:
        mask_path = os.path.join(f"{ham10000_dataset_path}/HAM10000_masks/", f"{img_id}_segmentation.png")
        image_path = os.path.join(f"{ham10000_dataset_path}/HAM10000_images", f"{img_id}.jpg")
        distance_path = os.path.join(f"{ham10000_dataset_path}/HAM10000_distance_maps", f"{img_id}_distance_map.png")
        if os.path.exists(mask_path) and os.path.exists(image_path) and os.path.exists(distance_path):
          mask_paths.append(mask_path)
          img_paths.append(image_path)
          distance_paths.append(distance_path)

    assert len(img_paths) == len(mask_paths) == len(distance_paths) == len(image_ids)

    return img_paths, mask_paths, distance_paths

def load_HAM10000():
  '''
  Returns the image IDs to load images
  Returns the labels for the images
  Returns the patient IDs for the images to avoid data leakage
  '''

  # HAM10000 Dataset
  ham10000_df = pd.read_csv(f"{ham10000_dataset_path}/HAM10000_metadata.csv")

  # Get all the different patient labels
  patient_labels = ham10000_df.groupby("lesion_id")["dx"].first().reset_index()

  # Split based on unique skin lesion IDs
  # Stratify based on the skin lesion type (dx) to have equal balance size
  train_patients, test_patients = train_test_split(
      patient_labels["lesion_id"].values, test_size=0.2, stratify=patient_labels["dx"].values, random_state=42
  )

  # Get all the training data
  train_df = ham10000_df[ham10000_df["lesion_id"].isin(train_patients)]

  train_image_ids = train_df["image_id"].values
  train_dx = train_df["dx"].values
  train_lesion_ids = train_df["lesion_id"].values

  test_df = ham10000_df[ham10000_df["lesion_id"].isin(test_patients)]
  test_image_ids = test_df["image_id"].values
  test_dx = test_df["dx"].values
  test_lesion_ids = test_df["lesion_id"].values

  return train_image_ids, train_dx, train_lesion_ids, test_image_ids, test_dx, test_lesion_ids

Prepares the dataset for use in TF

In [ ]:
def segmentation_load_dataset(img_path, mask_path, distance_path):
    '''
    Loads the image, mask, and distance maps
    img_path: path to image
    mask_path: path to mask
    distance_path: path to distance map
    returns: image, mask, distance map
    '''
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)       # decode JPG as RGB (3 channels)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    img.set_shape([IMG_SIZE[0], IMG_SIZE[1], 3])

    # MASK (PNG, already grayscale)
    m = tf.io.read_file(mask_path)
    m = tf.image.decode_png(m, channels=1)
    m = tf.image.resize(m, IMG_SIZE, method='nearest')
    m = tf.cast(m > 127, tf.float32)                  # binarize mask (0 or 1)
    m.set_shape([IMG_SIZE[0], IMG_SIZE[1], 1])

    # DISTANCE MAP
    d = tf.io.read_file(distance_path)
    d = tf.image.decode_png(d, channels=1)
    d = tf.image.resize(d, IMG_SIZE, method='nearest')
    d = tf.cast(d, tf.float32) / 255.0
    d = d + 1
    d.set_shape([IMG_SIZE[0], IMG_SIZE[1], 1])

    return img, m, d

def segmentation_augment(img, mask, dist):
    # Random horizontal flip
    if tf.random.uniform(()) > 0.5:
        img  = tf.image.flip_left_right(img)
        mask = tf.image.flip_left_right(mask)
        dist = tf.image.flip_left_right(dist)

    if tf.random.uniform(()) > 0.5:
        img  = tf.image.flip_up_down(img)
        mask = tf.image.flip_up_down(mask)
        dist = tf.image.flip_up_down(dist)

    if tf.random.uniform(()) > 0.5:
        img  = tf.image.random_brightness(img, max_delta=0.1)

    if tf.random.uniform(()) > 0.5:
        img  = tf.image.random_contrast(img, lower=0.9, upper=1.1)

    if tf.random.uniform(()) > 0.5:
        img = tf.image.rot90(img)
        mask  = tf.image.rot90(mask)
        dist  = tf.image.rot90(dist)

    return img, mask, dist

def format_deep_supervision(image, mask, dist):
    # Downsample the mask for each decoder block
    m2 = tf.image.resize(mask, (64, 64), method='nearest')
    m3 = tf.image.resize(mask, (32, 32), method='nearest')
    m4 = tf.image.resize(mask, (16, 16), method='nearest')

    masks = {
        "o1": mask,  # 128x128
        "o2": m2,    # 64x64
        "o3": m3,    # 32x32
        "o4": m4,    # 16x16
    }

    dist2 = tf.image.resize(dist, (64, 64), method='nearest')
    dist3 = tf.image.resize(dist, (32, 32), method='nearest')
    dist4 = tf.image.resize(dist, (16, 16), method='nearest')

    dists = {
        "o1": dist,  # 128x128
        "o2": dist2, # 64x64
        "o3": dist3, # 32x32
        "o4": dist4, # 16x16
    }

    return image, masks, dists

Make the straifier and k fold

In [ ]:
# Stratifier
sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

## Preprocessing

Smoothing Filter

In [ ]:
def smooth_filter(image, ksize):
    lab_image = cv2.cvtColor(image, cv2.COLOR_BGR2YUV)
    ch1 = lab_image[:, :, 0]
    ch2 = lab_image[:, :, 1]
    ch3 = lab_image[:, :, 2]

    # ch1 = cv2.GaussianBlur(ch1, (ksize, ksize), 0)
    ch1 = cv2.blur(ch1, (ksize, ksize))
    # ch1 = cv2.medianBlur(ch1, ksize=ksize)
    median_image =  cv2.cvtColor(cv2.merge([ch1.astype(np.uint8), ch2.astype(np.uint8), ch3.astype(np.uint8)]), cv2.COLOR_YUV2BGR)
    return median_image.astype(np.uint8)

CLAHE

In [ ]:
def equalize_channel(image, grid, limit=2.0):
    lab_image = cv2.cvtColor(image, cv2.COLOR_BGR2YUV)
    ch1 = lab_image[:, :, 0]
    ch2 = lab_image[:, :, 1]
    ch3 = lab_image[:, :, 2]

    clahe = cv2.createCLAHE(clipLimit=limit, tileGridSize=(grid, grid))
    ch1 = clahe.apply(ch1)
    clahe_image =  cv2.cvtColor(cv2.merge([ch1.astype(np.uint8), ch2.astype(np.uint8), ch3.astype(np.uint8)]), cv2.COLOR_YUV2BGR)

    return clahe_image

Hair Removal Algorithm

In [ ]:
def dull_razor(image, kernel):
    """
    Remove hairs from RGB image using the 'dull razor' method.
    Input: image as numpy array, RGB, uint8
    Output: processed image, same shape, uint8
    """
    # Convert RGB → BGR for OpenCV
    gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Blackhat morphological operation
    blackhat = cv2.morphologyEx(gray_image, cv2.MORPH_BLACKHAT, kernel=kernel)

    # Closing
    closing = cv2.morphologyEx(blackhat, cv2.MORPH_CLOSE, kernel=kernel)

    # Threshold
    threshold = cv2.threshold(blackhat, 20, 255, cv2.THRESH_BINARY)

    # Inpaint
    hairless_image = cv2.inpaint(image, threshold[1], 6, cv2.INPAINT_TELEA)

    return hairless_image

## Postprocessing

In [ ]:
def conn_components(image, connectivity):
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(image, connectivity=connectivity)
    areas = stats[1:, cv2.CC_STAT_AREA]
    centroids = centroids[1: ]

    # Filter for area and how close it is to the center
    average_area = np.median(areas)
    area_valid_components = np.argwhere(areas >= average_area).flatten() + 1

    center = np.array([image.shape[1] // 2, image.shape[0] // 2])
    distances = np.linalg.norm(centroids - center, axis=1)
    average_distance = np.median(distances)
    distance_valid_components = np.argwhere(distances <= average_distance).flatten() + 1

    common_components = np.intersect1d(area_valid_components, distance_valid_components)

    conn_image = np.where(np.isin(labels, common_components), 255, 0).astype(np.uint8)

    return conn_image

## Metrics

In [ ]:
def dice_np(y_true, y_pred, threshold=0.5, smooth=1e-7):
    y_true = (y_true > threshold)
    y_pred = (y_pred > threshold)

    intersection = np.logical_and(y_true, y_pred).sum()
    return (2. * intersection + smooth) / (y_true.sum() + y_pred.sum() + smooth)


def iou_np(y_true, y_pred, threshold=0.5, smooth=1e-7):
    y_true = (y_true > threshold)
    y_pred = (y_pred > threshold)

    intersection = np.logical_and(y_true, y_pred).sum()
    union = np.logical_or(y_true, y_pred).sum()
    return (intersection + smooth) / (union + smooth)

# Segmentation

In this section, we investigate different segmentation based models. This includes Otsu thresholding, EM clustering, and the UNET segmentation architecture.

### Otsu Segmentation

In [ ]:
def otsu_thresholding(image):
  '''
  Image is a one channel image used for Otsu Thresholding
  '''
  _ , otsu_image = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
  otsu_image = cv2.bitwise_not(otsu_image)

  return otsu_image

Iterate through the training set

In [ ]:
train_image_ids, _, _, _, _, _ = load_HAM10000()
training_images, training_masks, _ = get_HAM10000_paths(train_image_ids)

# Load stuff so that I/O is not swamped
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
cross_kernel = cv2.getStructuringElement(cv2.MORPH_CROSS, (17, 17))

dice = 0.0
IoU = 0.0

for i in tqdm(range(len(training_images))):
  img = cv2.imread(training_images[i])
  img = dull_razor(img, cross_kernel)
  gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
  gray = cv2.resize(gray, (128, 128))

  mask = otsu_thresholding(gray)
  mask = cv2.resize(mask, (600, 450), interpolation=cv2.INTER_NEAREST_EXACT).astype(np.uint8)
  mask = conn_components(mask, connectivity=4)
  mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

  groundtruth_mask = cv2.imread(training_masks[i], cv2.IMREAD_GRAYSCALE)

  dice += dice_np(groundtruth_mask, mask)
  IoU += iou_np(groundtruth_mask, mask)

dice /= len(training_images)
IoU /= len(training_images)

print(f"Average Dice: {dice}")
print(f"Average IoU: {IoU}")

## EM Segmentation

Create feature vector

In [ ]:
def create_feature_vector(image):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)

    ch1 = image[:,:, 0].astype(np.float64).flatten()
    ch2 = image[:,:, 1].astype(np.float64).flatten()
    ch3 = image[:,:, 2].astype(np.float64).flatten()

    feature_vectors = np.stack([ch1, ch2, ch3], axis=1)

    scaler = StandardScaler()
    feature_vectors = scaler.fit_transform(feature_vectors)

    return feature_vectors

Perform EM Clustering

In [ ]:
from sklearn.mixture import GaussianMixture

def em_clustering(image, features, circle_mask, height, width):
    # Flatten the mask
    # mask_flat = circle_mask.flatten().astype(bool)

    # Select only features inside mask
    # masked_features = features[mask_flat]

    gmm = GaussianMixture(n_components=2, covariance_type='full', random_state=42)
    gmm.fit(features)

    labels = gmm.predict(features)
    # labels = np.full(mask_flat.shape, fill_value=-1, dtype=np.int32)  # -1 for ignored pixels
    # labels[mask_flat] = masked_labels
    labels = labels.reshape((height, width))

    # count = np.count_nonzero(labels == -1)
    # print(labels.shape)

    # Identify darker cluster based on first channel
    cluster_centers = gmm.means_
    darker_cluster = np.argmin(cluster_centers[:,2])

    # Build final EM image
    em_image = np.zeros((height, width), dtype=np.uint8)
    em_image[labels == darker_cluster] = 255  # only keep darker cluster inside mask

    return em_image

EM Clustering

In [ ]:
from tqdm import tqdm
import numpy as np
import cv2

train_image_ids, _, _, _, _, _ = load_HAM10000()
training_images, training_masks, _ = get_HAM10000_paths(train_image_ids)

from tqdm import tqdm

# Load stuff so that I/O is not swamped
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
hair_kernel = cv2.getStructuringElement(cv2.MORPH_CROSS, (17, 17))

dice = 0.0
IoU = 0.0

for i in tqdm(range(len(training_images))):
  # Load the image
  skin_lesion_image = cv2.imread(training_images[i])
  skin_lesion_image = dull_razor(skin_lesion_image, kernel=hair_kernel)
  skin_lesion_image = cv2.resize(skin_lesion_image, (128, 128))

  features = create_feature_vector(skin_lesion_image)
  em_image = em_clustering(skin_lesion_image, features, None, 128, 128)
  em_image = cv2.resize(em_image, (600, 450), interpolation=cv2.INTER_NEAREST).astype(np.uint8)

  # Connected Components
  em_image = conn_components(em_image, connectivity=4)

  # Morphological Operations
  kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
  em_image = cv2.morphologyEx(em_image, cv2.MORPH_CLOSE, kernel)

  groundtruth_mask = cv2.imread(training_masks[i], cv2.IMREAD_GRAYSCALE)

  dice += dice_np(groundtruth_mask, mask)
  IoU += iou_np(groundtruth_mask, mask)

dice /= len(training_images)
IoU /= len(training_images)

print(f"Average Dice: {dice}")
print(f"Average IoU: {IoU}")

## U-Net

### Loss Functions

In [ ]:
def dice_loss(y_true, y_pred, smooth=1e-6):
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1, 2, 3])
    sums = tf.reduce_sum(y_true, axis=[1, 2, 3]) + tf.reduce_sum(y_pred, axis=[1, 2, 3])
    dice = (2.0 * intersection + smooth) / (sums + smooth)
    return 1.0 - tf.reduce_mean(dice)

def iou_loss(y_true, y_pred, smooth=1e-6):
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1,2,3])  # sum over H, W, C
    union = tf.reduce_sum(y_true, axis=[1,2,3]) + tf.reduce_sum(y_pred, axis=[1,2,3]) - intersection
    iou = (intersection + smooth) / (union + smooth)
    return 1.0 - tf.reduce_mean(iou)  # average IoU across batch

def bce_iou_loss(y_true, y_pred):
    bce = tf.reduce_mean(tf.keras.losses.binary_crossentropy(y_true, y_pred))
    iou = iou_loss(y_true, y_pred)
    beta = 0.25
    return beta * bce + (1 - beta) * iou

def bce_dice_loss(y_true, y_pred):
    bce = tf.reduce_mean(tf.keras.losses.binary_crossentropy(y_true, y_pred))
    dice = dice_loss(y_true, y_pred)
    return bce + tf.math.log(tf.math.cosh(dice))

TF evaluation metrics that will be used to monitor the IoU and dice while training

In [ ]:
def hard_dice(y_true, y_pred, threshold=0.5, smooth=1e-6):
    # Ensure float32
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    # Apply threshold to predictions
    y_pred = tf.cast(y_pred > threshold, tf.float32)

    # Compute intersection and union per image
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1,2,3])
    union = tf.reduce_sum(y_true, axis=[1,2,3]) + tf.reduce_sum(y_pred, axis=[1,2,3])

    # Dice coefficient per image
    dice = (2 * intersection + smooth) / (union + smooth)

    # Return mean Dice over batch
    return tf.reduce_mean(dice)

def hard_iou(y_true, y_pred, threshold=0.5, smooth=1e-6):
    # Ensure float32
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    # Apply threshold to predictions
    y_pred = tf.cast(y_pred > threshold, tf.float32)

    # Compute intersection and union per image
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1,2,3])
    union = tf.reduce_sum(y_true, axis=[1,2,3]) + tf.reduce_sum(y_pred, axis=[1,2,3]) - intersection

    # IoU per image
    iou = (intersection + smooth) / (union + smooth)

    # Return mean IoU over batch
    return tf.reduce_mean(iou)

### Segmentation Training Pipeline

General training pipeline for training segmentation

In [ ]:
def train_segmentation_model(model, train_ds, val_ds, loss="binary_crossentropy", learning_rate=0.0001, epochs=20, deep_supervision=False):

  reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_hard_iou',
    factor=0.5,
    patience=3,
    min_delta=0.0001,
    cooldown=0,
    min_lr=0.000001,
    verbose=1,
    mode='max'
  )

  # Create optimizer
  optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

  # Create early stopping
  early_stopping = tf.keras.callbacks.EarlyStopping(
      monitor='val_hard_iou',
      min_delta=0.001,
      patience=9,
      verbose=1,
      mode='max',
      restore_best_weights=True
  )

  if deep_supervision:
    metrics = {"o1": [hard_dice, hard_iou]}
    loss_func = {"o1": loss,
                 "o2": loss,
                 "o3": loss,
                 "o4": loss}
  else:
    metrics = [hard_dice, hard_iou]
    loss_func = loss

  # Compile the model
  model.compile(
      optimizer=optimizer,
      loss=loss_func,
      metrics=metrics
  )

  # Fit the model
  model.fit(
      train_ds,
      validation_data=val_ds,
      epochs=epochs,
      callbacks=[early_stopping, reduce_lr]
  )

  return model

#### Initialize U-Net

In [ ]:
def unet_encoder_block(inputs, num_filters, residual=False, activation="relu"):
    x = tf.keras.layers.Conv2D(num_filters, 3, padding="same")(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation(activation)(x)
    x = tf.keras.layers.Conv2D(num_filters, 3, padding="same")(x)
    x = tf.keras.layers.BatchNormalization()(x)

    if residual:
      # Make the input the same size as x so we can add them together
      shortcut = tf.keras.layers.Conv2D(num_filters, 1, strides=1, padding='same')(inputs)
      x = tf.keras.layers.Add()([x, shortcut])

    x = tf.keras.layers.Activation(activation)(x)

    return x

def attention_gate(x, g, inter_channels, activation="relu"):
    """
    x: skip connection (encoder feature map)
    g: gating signal (decoder feature map, upsampled)
    """

    # Project skip and gate to lower-dimensional space
    theta_x = tf.keras.layers.Conv2D(inter_channels, 1, padding="same")(x)
    theta_x = tf.keras.layers.BatchNormalization()(theta_x)

    phi_g   = tf.keras.layers.Conv2D(inter_channels, 1, padding="same")(g)
    phi_g   = tf.keras.layers.BatchNormalization()(phi_g)

    # Combine
    f = tf.keras.layers.Add()([theta_x, phi_g])
    f = tf.keras.layers.Activation(activation)(f)

    # Produce attention coefficients
    psi = tf.keras.layers.Conv2D(1, 1, padding="same")(f)
    psi = tf.keras.layers.Activation("sigmoid")(psi)

    # Apply attention to skip connection
    out = tf.keras.layers.Multiply()([x, psi])

    return out

def unet_decoder_block(inputs, skip_feature, num_filters, residual=False, attention=False, activation="relu"):
    upsampled = tf.keras.layers.Conv2DTranspose(num_filters, 2, strides=2, padding="same")(inputs)

    if attention:
      skip_feature = attention_gate(
          x=skip_feature,
          g=upsampled,
          inter_channels=num_filters // 2
      )

    # Concatenate
    concatenate = tf.keras.layers.Concatenate()([upsampled, skip_feature])

    x = tf.keras.layers.Conv2D(num_filters, 3, padding="same")(concatenate)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation(activation)(x)
    x = tf.keras.layers.Conv2D(num_filters, 3, padding="same")(x)
    x = tf.keras.layers.BatchNormalization()(x)

    if residual:
      shortcut = tf.keras.layers.Conv2D(num_filters, 1, strides=1, padding='same')(concatenate)
      x = tf.keras.layers.Add()([x, shortcut])

    x = tf.keras.layers.Activation(activation)(x)

    return x

def unet_model(input_shape=(512, 512, 3), num_classes=1, deep_supervision=False, attention=False, residual=False, activation="relu"):
    inputs = tf.keras.layers.Input(shape=input_shape)

    e1 = unet_encoder_block(inputs, 64, residual=residual, activation=activation)
    e2 = unet_encoder_block(tf.keras.layers.MaxPool2D(pool_size=(2, 2), strides=2)(e1), 128, residual=residual, activation=activation)
    e3 = unet_encoder_block(tf.keras.layers.MaxPool2D(pool_size=(2, 2), strides=2)(e2), 256, residual=residual, activation=activation)
    e4 = unet_encoder_block(tf.keras.layers.MaxPool2D(pool_size=(2, 2), strides=2)(e3), 512, residual=residual, activation=activation)

    b = unet_encoder_block(tf.keras.layers.MaxPool2D(pool_size=(2, 2), strides=2)(e4), 1024, residual=residual, activation=activation)

    d4 = unet_decoder_block(b, e4, 512, residual=residual, attention=attention, activation=activation)
    d3 = unet_decoder_block(d4, e3, 256, residual=residual, attention=attention, activation=activation)
    d2 = unet_decoder_block(d3, e2, 128, residual=residual, attention=attention, activation=activation)
    d1 = unet_decoder_block(d2, e1, 64, residual=residual, attention=attention, activation=activation)

    outputs = tf.keras.layers.Conv2D(num_classes, 1, padding='same', activation='sigmoid')(d1)

    # For deep supervision
    if deep_supervision:
      o1 = tf.keras.layers.Conv2D(num_classes, 1, padding='same', activation='sigmoid', name="o1")(d1)
      o2 = tf.keras.layers.Conv2D(num_classes, 1, padding='same', activation='sigmoid', name="o2")(d2)
      o3 = tf.keras.layers.Conv2D(num_classes, 1, padding='same', activation='sigmoid', name="o3")(d3)
      o4 = tf.keras.layers.Conv2D(num_classes, 1, padding='same', activation='sigmoid', name="o4")(d4)

      outputs = {
          "o1": o1,
          "o2": o2,
          "o3": o3,
          "o4": o4,
      }

    model = tf.keras.models.Model(inputs=inputs, outputs=outputs, name="U-Net")
    print(f"Residual: {residual}")
    print(f"Attention: {attention}")
    print(f"Deep Supervision: {deep_supervision}")
    print(f"Activation: {activation}")
    return model

#### Train U-net

This is done with five-fold cross validation

In [ ]:
# Parameters to change
deep_supervision = False
residual = True
attention = True
activation = "swish"
batch = 64
loss = bce_iou_loss

train_image_ids, train_dx, train_lesion_ids, test_image_ids, test_dx, test_lesion_ids = load_HAM10000()

for fold, (train_idx, val_idx) in enumerate(sgkf.split(train_image_ids, train_dx, train_lesion_ids)):
    print(f"\n===== Fold {fold+1} =====")

    # Gets the image ID for image paths
    train_ids, val_ids = train_image_ids[train_idx], train_image_ids[val_idx]

    # Get the corresponding image and train masks
    train_imgs, train_masks, train_dist = get_HAM10000_paths(train_ids)
    val_imgs, val_masks, val_dist = get_HAM10000_paths(val_ids)

    # Get the training and validation datasets
    train_ds = (
        tf.data.Dataset.from_tensor_slices((train_imgs, train_masks, train_dist))
        .map(segmentation_load_dataset, num_parallel_calls=AUTOTUNE)
        .map(segmentation_augment, num_parallel_calls=AUTOTUNE)
        .shuffle(2000)
        .batch(batch)
        .prefetch(AUTOTUNE)
    )

    val_ds = (
        tf.data.Dataset.from_tensor_slices((val_imgs, val_masks, val_dist))
        .map(segmentation_load_dataset, num_parallel_calls=AUTOTUNE)
        .batch(batch)
        .prefetch(AUTOTUNE)
    )

    if deep_supervision:
      train_ds = train_ds.map(format_deep_supervision, num_parallel_calls=AUTOTUNE)
      val_ds = val_ds.map(format_deep_supervision, num_parallel_calls=AUTOTUNE)

    # Create the model
    model = unet_model(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3), num_classes=1,
                          deep_supervision=deep_supervision,
                          attention=attention,
                          residual=residual,
                          activation=activation)

    # Fit the model
    fitted_model = train_segmentation_model(model, train_ds, val_ds, epochs=200, learning_rate=0.0001, deep_supervision=deep_supervision, loss=loss)

    # Get validation segmentation masks based on the fitted model
    validation_masks = fitted_model.predict(val_ds)

    # Get the original masks
    original_images, groundtruth_masks, _ = get_HAM10000_paths(val_ids)

    dice = 0.0
    IoU = 0.0

    for i in range(len(val_ids)):
      # Get the groundtruth mask and normalize between 0-1
      # Color Image

      groundtruth_mask = groundtruth_masks[i]
      groundtruth_mask = cv2.imread(groundtruth_mask, cv2.IMREAD_GRAYSCALE) / 255.0

      # Resize the validation masks
      mask = validation_masks[i]
      mask = cv2.resize(mask, (600, 450))
      mask = (mask > 0.5).astype("uint8") * 255

      # Calculate IoU and dice
      dice += dice_np(groundtruth_mask, mask)
      IoU += iou_np(groundtruth_mask, mask)

    dice /= len(val_ids)
    IoU /= len(val_ids)

    print(f"Average Dice: {dice}")
    print(f"Average IoU: {IoU}")

Train on all data for final model (no five-fold cross-validation)

In [ ]:
# Best performing model parameters:
deep_supervision = False
residual = True
attention = True
activation = "swish"
batch = 64
loss = bce_iou_loss

train_image_ids, train_dx, train_lesion_ids, test_image_ids, test_dx, test_lesion_ids = load_HAM10000()

# Get the corresponding image and train masks
train_imgs, train_masks, train_dist = get_HAM10000_paths(train_image_ids)
val_imgs, val_masks, val_dist = get_HAM10000_paths(test_image_ids)

# Get the training and validation datasets
train_ds = (
    tf.data.Dataset.from_tensor_slices((train_imgs, train_masks, train_dist))
    .map(segmentation_load_dataset, num_parallel_calls=AUTOTUNE)
    .map(segmentation_augment, num_parallel_calls=AUTOTUNE)
    .shuffle(2000)
    .batch(batch)
    .prefetch(AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((val_imgs, val_masks, val_dist))
    .map(segmentation_load_dataset, num_parallel_calls=AUTOTUNE)
    .batch(batch)
    .prefetch(AUTOTUNE)
)

if deep_supervision:
  train_ds = train_ds.map(format_deep_supervision, num_parallel_calls=AUTOTUNE)
  val_ds = val_ds.map(format_deep_supervision, num_parallel_calls=AUTOTUNE)

# Create the model
model = unet_model(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3), num_classes=1,
                      deep_supervision=deep_supervision,
                      attention=attention,
                      residual=residual,
                      activation=activation)

# Fit the model
fitted_model = train_segmentation_model(model, train_ds, val_ds=val_ds, epochs=200, learning_rate=0.0001, deep_supervision=deep_supervision, loss=loss)
fitted_model.evaluate(val_ds)
fitted_model.save(f'./segmentation_model.keras')

#### Postprocessing

Morphological Operations

In [ ]:
train_image_ids, train_dx, train_lesion_ids, test_image_ids, test_dx, test_lesion_ids = load_HAM10000()

# Get the corresponding image and train masks
train_imgs, train_masks, train_dist = get_HAM10000_paths(train_image_ids)
val_imgs, val_masks, val_dist = get_HAM10000_paths(test_image_ids)

val_ds = (
    tf.data.Dataset.from_tensor_slices((val_imgs, val_masks, val_dist))
    .map(segmentation_load_dataset, num_parallel_calls=AUTOTUNE)
    .batch(batch)
    .prefetch(AUTOTUNE)
)

segmentation_model_path = f'./segmentation_model.keras'
segmentation_model = tf.keras.models.load_model(segmentation_model_path,
                                                custom_objects={"bce_iou_loss": bce_iou_loss,
                                                                "hard_dice": hard_dice,
                                                                "hard_iou": hard_iou})

validation_masks = segmentation_model.predict(val_ds)

for kernel_size in [3, 7, 11]:
  dice = 0.0
  IoU = 0.0

  for i in tqdm(range(len(val_ids))):
    # Postpress each mask and get its corresponding feature
    cur_mask = cv2.resize(validation_masks[i], (600, 450))
    cur_mask = (cur_mask > 0.5).astype(np.uint8) * 255

    # Gaussian blur
    cur_mask = cv2.GaussianBlur(cur_mask, (kernel_size, kernel_size), 0) / 255.0

    groundtruth_mask = cv2.imread(val_masks[i], cv2.IMREAD_GRAYSCALE) / 255.0

    # Calculate dice and IoU
    dice += dice_np(groundtruth_mask, cur_mask)
    IoU += iou_np(groundtruth_mask, cur_mask)

  dice /= len(val_ids)
  IoU /= len(val_ids)

  print(f"Kernel: {kernel_size}")
  print(f"Average Dice: {dice}")
  print(f"Average IoU: {IoU}")

# ABCDE-Based Feature Extraction

In this section, we perform ABCDE-based feature extraction based on segmentation masks produced for UNET segmentation since this model provided us the best performance.

## Helper Functions

Functions calculate IoU, lesion border, area, and perimeter

In [ ]:
def calculate_iou(mask, groundtruth_mask):
    mask = (mask > 0).astype(np.uint8)  # Now contains 0s and 1s
    groundtruth_mask = (groundtruth_mask > 0).astype(np.uint8)

    overlap = np.logical_and(mask, groundtruth_mask)
    union = np.logical_or(mask, groundtruth_mask)

    iou = np.sum(overlap) / np.sum(union)

    return iou

def get_max_contour(mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    max_contour = max(contours, key=cv2.contourArea)
    return max_contour

def get_contour_area(contour):
    return cv2.contourArea(contour)

def get_contour_perimeter(contour):
    return cv2.arcLength(contour, True)

## Asymmetry

Roundness

In [ ]:
def roundness(area, perimeter):
    roundness_score = (4 * np.pi  * area) / (perimeter * perimeter)
    return roundness_score

Symmetry Distance

In [ ]:
def symmetry_distance(mask, contour, debug):
    n = len(contour)

    # Separate the x and y coords
    points = contour.reshape(-1, 2)
    x_coords = points[:, 0]
    y_coords = points[:, 1]

    # Calculate the centroid
    M = cv2.moments(contour)
    cx = int(M['m10'] / M['m00'])
    cy = int(M['m01'] / M['m00'])
    center = np.array([cx, cy])

    # 1) Folding
    folded_x_coords = []
    folded_y_coords = []
    i = 0
    for x_coord, y_coord in zip(x_coords, y_coords):
        theta = (2 * np.pi * i) / n
        folded_x = center[0] + np.cos(theta) * (x_coord - center[0]) - np.sin(theta) * (y_coord - center[1])
        folded_y = center[1] + np.sin(theta) * (x_coord - center[0]) + np.cos(theta) * (y_coord - center[1])
        folded_x_coords.append(folded_x)
        folded_y_coords.append(folded_y)
        i += 1

    folded_x_coords = np.array(folded_x_coords, np.float32)
    folded_y_coords = np.array(folded_y_coords, np.float32)

    # 2) Plot P0
    p0_x = np.mean(folded_x_coords)
    p0_y = np.mean(folded_y_coords)

    # 3) Unfold
    unfolded_x_coords = []
    unfolded_y_coords = []
    i = 0
    while i < n:
        theta = (-2 * np.pi * i) / n
        unfolded_x = center[0] + np.cos(theta) * (p0_x - center[0]) - np.sin(theta) * (p0_y - center[1])
        unfolded_y = center[1] + np.sin(theta) * (p0_x - center[0]) + np.cos(theta) * (p0_y - center[1])
        unfolded_x_coords.append(unfolded_x)
        unfolded_y_coords.append(unfolded_y)
        i += 1

    unfolded_x_coords = np.array(unfolded_x_coords, np.float32)
    unfolded_y_coords = np.array(unfolded_y_coords, np.float32)

    # 4) Calcualte the symmetry distance
    sd = np.sum((x_coords - unfolded_x_coords) ** 2 + (y_coords - unfolded_y_coords) ** 2) / n

    if debug:
        fig, ax = plt.subplots(figsize=(6, 4))

        ax.plot(x_coords, y_coords, 'go-', markersize=1)  # Connects in order
        ax.plot([x_coords[-1], x_coords[0]], [y_coords[-1], y_coords[0]], 'go-', label="Contour", markersize=1)  # Closes the triangle manually

        ax.plot(folded_x_coords, folded_y_coords, 'bo-', markersize=1)  # Connects in order
        ax.plot([folded_x_coords[-1], folded_x_coords[0]], [folded_y_coords[-1], folded_y_coords[0]], 'bo-', label="Folded Shape", markersize=1)  # Closes the triangle manually

        ax.plot(unfolded_x_coords, unfolded_y_coords, 'mo-', markersize=1)  # Connects in order
        ax.plot([unfolded_x_coords[-1], unfolded_x_coords[0]], [unfolded_y_coords[-1], unfolded_y_coords[0]], 'mo-', label="Unfolded Shape", markersize=1)  # Closes the triangle manually

        ax.plot(p0_x, p0_y, '*', label="P0", markersize=8, color="gold")

        ax.set_xlim(0, 600)
        ax.set_ylim(450, 0)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.legend(loc="upper right")

        plt.tight_layout()
        plt.show()

    return sd

Reflection Across Principle Axis

In [ ]:
def rotated_image(theta, x, y):
    # Principal Axis Rotation
    rotated_x = np.cos(-theta) * (x) - np.sin(-theta) * (y)
    rotated_y = np.sin(-theta) * (x) + np.cos(-theta) * (y)

    x_min = rotated_x.min()
    y_min = rotated_y.min()

    bounded_x = (rotated_x - x_min).astype(np.int32)
    bounded_y = (rotated_y - y_min).astype(np.int32)

    new_contour = np.stack((bounded_x, bounded_y), axis=1)
    x_min, x_max = bounded_x.min(), bounded_x.max()
    y_min, y_max = bounded_y.min(), bounded_y.max()

    new_mask = np.zeros(shape=(y_max - y_min, x_max - x_min))  # single-channel
    cv2.drawContours(new_mask, [new_contour], -1, color=255, thickness=cv2.FILLED)

    return new_mask

def principle_axis(mask, contour):

    # Get the x and y coords
    points = contour.reshape(-1, 2)
    x_coords = points[:, 0]
    y_coords = points[:, 1]

    # Calculate the center of the centroid
    M = cv2.moments(contour)
    cx, cy = M['m10']/M['m00'], M['m01']/M['m00']

    # Calculate the orientation of the major axis relative to the horizontal axis
    Ixx, Iyy, Ixy = M['mu20'], M['mu02'], M['mu11']
    major_axis_theta = 0.5 * np.arctan2(2.0 * Ixy, Ixx - Iyy)   # radians

    # Center lesion at (0, 0)
    translated_x = x_coords - cx
    translated_y = y_coords - cy

    rotated_mask = rotated_image(major_axis_theta, translated_x, translated_y)

    major_flipped_mask = cv2.flip(rotated_mask, 0)
    minor_flipped_mask = cv2.flip(rotated_mask, 1)

    major_axis_iou = calculate_iou(major_flipped_mask, rotated_mask)
    minor_axis_iou = calculate_iou(minor_flipped_mask, rotated_mask)

    return major_axis_iou, minor_axis_iou

## Border Irregularities

Minimum Enscribed Circle

In [ ]:
def best_fit_circle(mask, contour, debug):

    # Fits the best ellipse to the contour
    center, radius = cv2.minEnclosingCircle(contour)

    circle_area = np.pi * (radius * radius)
    contour_area = cv2.contourArea(contour)
    circle_area_ratio = abs(1.0 - circle_area / contour_area)

    # Calculates assymmetry score based on iou
    h, w = mask.shape[:2]
    filled_contour = np.zeros((h, w), dtype=np.uint8)
    filled_circle = np.zeros((h, w), dtype=np.uint8)

    cv2.drawContours(filled_contour, [contour], 0, 255, -1)
    cv2.circle(filled_circle, (int(center[0]), int(center[1])), int(radius), 255, thickness=-1)
    circle_iou = calculate_iou(filled_contour, filled_circle)

    if debug:
        # Print the feature values
        print(f"Circle Area Ratio: {circle_area_ratio}")
        print(f"Circle IoU: {circle_iou}")
        # --- Fit minimum enclosing circle ---
        fig, ax1 = plt.subplots(figsize=(10, 5))

        t = np.linspace(0, 2*np.pi, 300)

        (center_x, center_y), radius = cv2.minEnclosingCircle(contour)
        x_circle = center_x + radius * np.cos(t)
        y_circle = center_y + radius * np.sin(t)

        # Right: circle
        ax1.imshow(mask)
        ax1.plot(x_circle, y_circle, 'b-', linewidth=2, label='Min Enclosing Circle')
        ax1.scatter(center_x, center_y, marker='o', s=100, color='blue', label='Circle Center')
        ax1.axis('off')
        ax1.set_title('Minimum Enclosing Circle')
        ax1.legend()

        plt.tight_layout()
        plt.show()


    return circle_area_ratio, circle_iou

Convex Hull

In [ ]:
def convex_hull(mask, contour, debug):
    # Get the convex hull
    hull = cv2.convexHull(contour)

    # Get the hull area and the contour area
    hull_area = cv2.contourArea(hull)
    contour_area = cv2.contourArea(contour)
    convex_area_ratio = abs(1.0 - hull_area / contour_area)

    h, w = mask.shape[:2]
    filled_contour = np.zeros((h, w), dtype=np.uint8)
    filled_convex = np.zeros((h, w), dtype=np.uint8)

    cv2.drawContours(filled_contour, [contour], 0, 255, -1)
    cv2.drawContours(filled_convex, [hull], 0, 255, -1)
    convex_iou = calculate_iou(filled_contour, filled_convex)

    if debug:
        print(f"Convex Area Ratio: {convex_area_ratio}")
        print(f"Convex IoU: {convex_iou}")

        plt.imshow(mask)
        plt.plot(contour[:,0,0], contour[:,0,1], 'g-', label='Contour')  # green
        hull_points = hull[:,0,:]
        plt.plot(np.append(hull_points[:,0], hull_points[0,0]),
                np.append(hull_points[:,1], hull_points[0,1]), 'r-', label='Convex Hull')  # red
        plt.legend()
        plt.axis("off")
        plt.show()

    return convex_area_ratio, convex_iou

## Color Features

Color Entropy

In [ ]:
from skimage.measure import shannon_entropy

def color_entropy(image, mask, color_space):
    if color_space == cv2.COLOR_BGR2HSV_FULL:
        hsv_image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV_FULL)
        hsv_image = hsv_image[mask == 255]
        hue_channel = hsv_image[:, 0].astype(np.float64).flatten()
        scaled_input = (2 * np.pi * hue_channel) / 255

        ch1_entropy = shannon_entropy((127.5 * (np.sin(scaled_input) + 1)).flatten())
        ch2_entropy = shannon_entropy(hsv_image[:, 1].astype(np.float64).flatten())
        ch3_entropy = shannon_entropy(hsv_image[:, 2].astype(np.float64).flatten())

        return ch1_entropy, ch2_entropy, ch3_entropy

    image = cv2.cvtColor(image, color_space)
    skin_lesion_image = image[mask == 255]
    ch1_entropy = shannon_entropy(skin_lesion_image[:, 0].flatten())
    ch2_entropy = shannon_entropy(skin_lesion_image[:, 1].flatten())
    ch3_entropy = shannon_entropy(skin_lesion_image[:, 2].flatten())

    return ch1_entropy, ch2_entropy, ch3_entropy

Color Standard Deviation

In [ ]:
def color_std(image, mask, color_space):
    if color_space == cv2.COLOR_BGR2HSV_FULL:
        hsv_image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV_FULL)
        hsv_image = hsv_image[mask == 255]
        hue_channel = hsv_image[:, 0].astype(np.float64).flatten()
        scaled_input = (2 * np.pi * hue_channel) / 255

        ch1_std = (127.5 * (np.sin(scaled_input) + 1)).flatten().std()
        ch2_std = hsv_image[:, 1].astype(np.float64).flatten().std()
        ch3_std = hsv_image[:, 2].astype(np.float64).flatten().std()

        return ch1_std, ch2_std, ch3_std

    image = cv2.cvtColor(image, color_space)
    skin_lesion_image = image[mask == 255]
    ch1_std = skin_lesion_image[:, 0].flatten().std()
    ch2_std = skin_lesion_image[:, 1].flatten().std()
    ch3_std = skin_lesion_image[:, 2].flatten().std()

    return ch1_std, ch2_std, ch3_std

Color Skew

In [ ]:
from scipy.stats import skew

def color_skew(image, mask, color_space):
    if color_space == cv2.COLOR_BGR2HSV_FULL:
        hsv_image = cv2.cvtColor(image, color_space)
        hsv_image = hsv_image[mask == 255]
        hue_channel = hsv_image[:, 0].astype(np.float64).flatten()
        scaled_input = (2 * np.pi * hue_channel) / 255

        ch1_skew = skew((127.5 * (np.sin(scaled_input) + 1)).flatten())
        ch2_skew = skew(hsv_image[:, 1].astype(np.float64).flatten())
        ch3_skew = skew(hsv_image[:, 2].astype(np.float64).flatten())

        return ch1_skew, ch2_skew, ch3_skew

    image = cv2.cvtColor(image, color_space)
    skin_lesion_image = image[mask == 255]
    ch1_skew = skew(skin_lesion_image[:, 0].flatten())
    ch2_skew = skew(skin_lesion_image[:, 1].flatten())
    ch3_skew = skew(skin_lesion_image[:, 2].flatten())

    return ch1_skew, ch2_skew, ch3_skew

Color Mean

In [ ]:
def color_mean(image, mask, color_space):
  if color_space == cv2.COLOR_BGR2HSV_FULL:
        hsv_image = cv2.cvtColor(image, color_space)
        hsv_image = hsv_image[mask == 255]
        hue_channel = hsv_image[:, 0].astype(np.float64).flatten()
        scaled_input = (2 * np.pi * hue_channel) / 255

        ch1_mean = np.mean((127.5 * (np.sin(scaled_input) + 1)).flatten())
        ch2_mean = np.mean(hsv_image[:, 1].astype(np.float64).flatten())
        ch3_mean = np.mean(hsv_image[:, 2].astype(np.float64).flatten())

        return ch1_mean, ch2_mean, ch3_mean

  image = cv2.cvtColor(image, color_space)
  skin_lesion_image = image[mask == 255]
  ch1_mean = np.mean(skin_lesion_image[:, 0].flatten())
  ch2_mean = np.mean(skin_lesion_image[:, 1].flatten())
  ch3_mean = np.mean(skin_lesion_image[:, 2].flatten())

  return ch1_mean, ch2_mean, ch3_mean

## Load ISIC 2018 Dataset

ISIC 2018

In [ ]:
isic2018_path = kagglehub.dataset_download("josephgalicinao/isic-2018-dataset")

print("Path to dataset files:", isic2018_path)

ISIC 2018 Test

In [ ]:
# Download latest version
isic2018_test_path = kagglehub.dataset_download("josephgalicinao/isic-2018-test-dataset")

print("Path to dataset files:", isic2018_test_path)

Get Dataset helper functions

In [ ]:
def get_train_isic2018():
  print("ISIC 2018 Training Dataset...")
  df = pd.read_csv(f"{isic2018_path}/metadata.csv")
  df = df.dropna(subset=['diagnosis_1'])

  isic_ids = df["isic_id"].values
  isic_dx = df["diagnosis_1"].map({'Malignant': 1, 'Benign': 0, 'Indeterminate': 1}).values
  isic_patient_ids = df["lesion_id"].values

  isic_paths = []
  for isic_id in isic_ids:
    isic_paths.append(f"{isic2018_path}/{isic_id}.jpg")

  assert len(isic_paths) == len(isic_dx) == len(isic_patient_ids)

  return isic_paths, isic_dx, isic_patient_ids

def get_test_isic2018():
  print("ISIC 2018 Test Dataset...")
  df = pd.read_csv(f"{isic2018_test_path}/metadata.csv")
  df = df.dropna(subset=['diagnosis_1'])

  isic_ids = df["isic_id"].values
  isic_dx = df["diagnosis_1"].map({'Malignant': 1, 'Benign': 0, 'Indeterminate': 1}).values

  isic_paths = []
  for isic_id in isic_ids:
    isic_paths.append(f"{isic2018_test_path}/{isic_id}.jpg")

  assert len(isic_paths) == len(isic_dx)

  return isic_paths, isic_dx

## Extract features

In [ ]:
def segmentation_load_dataset(img_path, y):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)   # or decode_png
    img = tf.image.resize(img, (128, 128))
    img = tf.cast(img, tf.float32) / 255.0

    return img, y

def get_features(skin_lesion_image, mask):

  try:
    mask = mask.astype(np.uint8)

    # Basic lesion features
    contour = get_max_contour(mask=mask)
    perimeter = get_contour_perimeter(contour)
    area = get_contour_area(contour)

    # -----Asymmetry-----
    circularity = roundness(area, perimeter)
    sd = symmetry_distance(mask, contour, debug=False)
    major_axis_iou, minor_axis_iou = principle_axis(mask, contour)

    # -----Border-----
    circle_iou, circle_area_ratio = best_fit_circle(mask, contour, debug=False)
    convex_hull_iou, convex_hull_area_ratio = convex_hull(mask, contour, debug=False)

    # -----Color-----
    red_std, green_std, blue_std = color_std(skin_lesion_image, mask, cv2.COLOR_BGR2RGB)
    l_std, a_std, b_std = color_std(skin_lesion_image, mask, cv2.COLOR_BGR2LAB)
    y_std, cr_std, cb_std = color_std(skin_lesion_image, mask, cv2.COLOR_BGR2YCR_CB)
    h_std, s_std, v_std = color_std(skin_lesion_image, mask, cv2.COLOR_BGR2HSV_FULL)

    red_entropy, green_entropy, blue_entropy = color_entropy(skin_lesion_image, mask, cv2.COLOR_BGR2RGB)
    l_entropy, a_entropy, b_entropy = color_entropy(skin_lesion_image, mask, cv2.COLOR_BGR2LAB)
    y_entropy, cr_entropy, cb_entropy = color_entropy(skin_lesion_image, mask, cv2.COLOR_BGR2YCR_CB)
    h_entropy, s_entropy, v_entropy = color_entropy(skin_lesion_image, mask, cv2.COLOR_BGR2HSV_FULL)

    red_skew, green_skew, blue_skew = color_skew(skin_lesion_image, mask, cv2.COLOR_BGR2RGB)
    l_skew, a_skew, b_skew = color_skew(skin_lesion_image, mask, cv2.COLOR_BGR2LAB)
    y_skew, cr_skew, cb_skew = color_skew(skin_lesion_image, mask, cv2.COLOR_BGR2YCR_CB)
    h_skew, s_skew, v_skew = color_skew(skin_lesion_image, mask, cv2.COLOR_BGR2HSV_FULL)

    red_mean, green_mean, blue_mean = color_mean(skin_lesion_image, mask, cv2.COLOR_BGR2RGB)
    l_mean, a_mean, b_mean = color_mean(skin_lesion_image, mask, cv2.COLOR_BGR2LAB)
    y_mean, cr_mean, cb_mean = color_mean(skin_lesion_image, mask, cv2.COLOR_BGR2YCR_CB)
    h_mean, s_mean, v_mean = color_mean(skin_lesion_image, mask, cv2.COLOR_BGR2HSV_FULL)

    features = [circularity, sd, major_axis_iou, minor_axis_iou,
                circle_iou, circle_area_ratio, convex_hull_iou, convex_hull_area_ratio,
                red_std, green_std, blue_std, l_std, a_std, b_std, y_std, cr_std, cb_std, h_std, s_std, v_std,
                red_entropy, green_entropy, blue_entropy, l_entropy, a_entropy, b_entropy, y_entropy, cr_entropy, cb_entropy, h_entropy, s_entropy, v_entropy,
                red_skew, green_skew, blue_skew, l_skew, a_skew, b_skew, y_skew, cr_skew, cb_skew, h_skew, s_skew, v_skew,
                red_mean, green_mean, blue_mean, l_mean, a_mean, b_mean, y_mean, cr_mean, cb_mean, h_mean, s_mean, v_mean]

    return features
  except:
    print("Error getting features")
    return [0] * 56

Get ABCDE features

In [ ]:
# Combined dataset

batch = 64
train_imgs, train_dx, train_patient_id = get_train_isic2018()
val_imgs, val_dx = get_test_isic2018()

train_ds = (
    tf.data.Dataset.from_tensor_slices((train_imgs, train_dx))
    .map(segmentation_load_dataset, num_parallel_calls=AUTOTUNE)
    .batch(batch)
    .prefetch(AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((val_imgs, val_dx))
    .map(segmentation_load_dataset, num_parallel_calls=AUTOTUNE)
    .batch(batch)
    .prefetch(AUTOTUNE)
)

batch = 64

segmentation_model_path = f'./segmentation_model.keras'
segmentation_model = tf.keras.models.load_model(segmentation_model_path,
                                                  custom_objects={"bce_iou_loss": bce_iou_loss,
                                                                  "hard_dice": hard_dice,
                                                                  "hard_iou": hard_iou})

with (open(f'./abcde_features_train.csv', 'w', newline='') as csvfile1):
  csv_writer1 = csv.writer(csvfile1)

  training_masks = segmentation_model.predict(train_ds)

  for i in tqdm(range(len(train_imgs)),
                      total=len(train_imgs),
                      desc="Getting automatic training ABCDE train features"):

    cur_mask = training_masks[i]
    cur_mask = (cur_mask > 0.5).astype(np.uint8) * 255
    cur_mask = cv2.resize(cur_mask, (600, 450))
    cur_image = cv2.imread(train_imgs[i])


    features = get_features(cur_image, cur_mask)

    csv_writer1.writerow([train_imgs[i], train_dx[i], train_patient_id[i]] + features)

with (open(f'./abcde_features_test.csv', 'w', newline='') as csvfile1):
  csv_writer1 = csv.writer(csvfile1)

  val_masks = segmentation_model.predict(val_ds)

  for i in tqdm(range(len(val_imgs)),
                      total=len(val_imgs),
                      desc="Getting automatic training ABCDE test features"):

    cur_mask = val_masks[i]
    cur_mask = (cur_mask > 0.5).astype(np.uint8) * 255
    cur_mask = cv2.resize(cur_mask, (600, 450))
    cur_image = cv2.imread(val_imgs[i])

    features = get_features(cur_image, cur_mask)

    csv_writer1.writerow([val_imgs[i], val_dx[i]] + features)

## Evaluation Metrics

In [ ]:
def get_metrics(y_val, y_pred):
  # y_true: true labels, y_scores: predicted probabilities
  precision, recall, thresholds = precision_recall_curve(y_val, y_pred)
  pr_auc = auc(recall, precision)
  precision = precision_score(y_val, y_pred, average='binary')
  recall = recall_score(y_val, y_pred, average='binary')
  f1 = f1_score(y_val, y_pred, average='binary')

  return [precision, recall, f1, pr_auc]

# Classical Machine Learning Classifiers

In this section, we take the ABCDE-based features and utilize different classical machine learning classifiers to distinguish between malignant and benign skin lesions.

### XGBoost

In [ ]:
def xgboost(x_train, y_train, x_val, y_val, n, skin_tones):
    xgb = XGBClassifier(n_estimators=n)
    xgb.fit(x_train, y_train)
    y_pred = xgb.predict(x_val)

    results = get_metrics(y_val, y_pred)

    return results

Fine-Tune

In [ ]:
# Import PCA
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
import pandas as pd
import csv

train_df = pd.read_csv(f'./abcde_features_train.csv')

dx = train_df.iloc[:, 1].values
patient_ids = train_df.iloc[:, 2].values
features = train_df.iloc[:, 3:].values

with open(f'./isic2018_xgboost_fine_tune.csv', 'w', newline='') as csvfile:
  csv_writer = csv.writer(csvfile)
  csv_writer.writerow(["Number Learners", "Precision", "Recall", "F1", "PR-AUC"])

  for n in [100, 200, 300, 400, 500]:
    for fold, (train_idx, val_idx) in enumerate(sgkf.split(features, dx, patient_ids)):
      train_features, val_features = features[train_idx], features[val_idx]
      train_dx, val_dx = dx[train_idx], dx[val_idx]

      # Scale the features
      scaler = StandardScaler()
      train_features = scaler.fit_transform(train_features)
      val_features = scaler.transform(val_features)

      results = xgboost(train_features, train_dx, val_features, val_dx, n=n, skin_tones=None)
      print(f"XGBoost: {n} | Precision: {results[0]} | Recall: {results[1]} | F1: {results[2]} | PR-AUC: {results[3]}")
      csv_writer.writerow([n] + results)

Evaluate (n = 200)

In [ ]:
# Import PCA
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
import pandas as pd
import csv

train_df = pd.read_csv(f'./abcde_features_train.csv')
val_df = pd.read_csv(f'./abcde_features_test.csv')

train_dx = train_df.iloc[:, 1].values
train_features = train_df.iloc[:, 3:].values
val_dx = val_df.iloc[:, 1].values
val_features = val_df.iloc[:, 2:].values

n = 200

# Scale the features
scaler = StandardScaler()
train_features = scaler.fit_transform(train_features)
val_features = scaler.transform(val_features)

results = xgboost(train_features, train_dx, val_features, val_dx, n=n, skin_tones=None)
print(f"XGBoost: {n} | Precision: {results[0]} | Recall: {results[1]} | F1: {results[2]} | PR-AUC: {results[3]}")

### KNN

In [ ]:
def knn(x_train, y_train, x_val, y_val, n):
    model = KNeighborsClassifier(n_neighbors=n)
    model.fit(x_train, y_train)
    y_pred = model.predict(x_val)

    results = get_metrics(y_val, y_pred)

    return results

Fine-Tune

In [ ]:
train_df = pd.read_csv(f'./abcde_features_train.csv')

dx = train_df.iloc[:, 1].values
patient_ids = train_df.iloc[:, 2].values
features = train_df.iloc[:, 3:].values

with open(f'./isic2018_knn_fine_tune.csv', 'w', newline='') as csvfile:
  csv_writer = csv.writer(csvfile)
  csv_writer.writerow(["Number Learners", "Precision", "Recall", "F1", "PR-AUC"])

  for n in [1, 2, 3, 4, 5]:
    for fold, (train_idx, val_idx) in enumerate(sgkf.split(features, dx, patient_ids)):
      train_features, val_features = features[train_idx], features[val_idx]
      train_dx, val_dx = dx[train_idx], dx[val_idx]

      # Scale the features
      scaler = StandardScaler()
      train_features = scaler.fit_transform(train_features)
      val_features = scaler.transform(val_features)

      results = knn(train_features, train_dx, val_features, val_dx, n=n)
      print(f"KNN: {n} | Precision: {results[0]} | Recall: {results[1]} | F1: {results[2]} | PR-AUC: {results[3]}")
      csv_writer.writerow([n] + results)

Evaluate (k = 5)

In [ ]:
train_df = pd.read_csv(f'./abcde_features_train.csv')
val_df = pd.read_csv(f'./abcde_features_test.csv')

train_dx = train_df.iloc[:, 1].values
train_features = train_df.iloc[:, 3:].values
val_dx = val_df.iloc[:, 1].values
val_features = val_df.iloc[:, 2:].values

n = 5

# Scale the features
scaler = StandardScaler()
train_features = scaler.fit_transform(train_features)
val_features = scaler.transform(val_features)

results = knn(train_features, train_dx, val_features, val_dx, n=n)
print(f"KNN: {n} | Precision: {results[0]} | Recall: {results[1]} | F1: {results[2]} | PR-AUC: {results[3]}")

### Random Forest

In [ ]:
def random_forest(x_train, y_train, x_val, y_val, n):
    rf = RandomForestClassifier(n_estimators=n, class_weight='balanced')
    rf.fit(x_train, y_train)
    y_pred = rf.predict(x_val)

    results = get_metrics(y_val, y_pred)

    return results

Fine-Tune

In [ ]:
train_df = pd.read_csv(f'./abcde_features_train.csv')

dx = train_df.iloc[:, 1].values
patient_ids = train_df.iloc[:, 2].values
features = train_df.iloc[:, 3:].values

with open(f'./isic2018_rf_fine_tune.csv', 'w', newline='') as csvfile:
  csv_writer = csv.writer(csvfile)
  csv_writer.writerow(["Number Learners", "Precision", "Recall", "F1", "PR-AUC"])

  for n in [100, 200, 300, 400, 500]:
    for fold, (train_idx, val_idx) in enumerate(sgkf.split(features, dx, patient_ids)):
      train_features, val_features = features[train_idx], features[val_idx]
      train_dx, val_dx = dx[train_idx], dx[val_idx]

      # Scale the features
      scaler = StandardScaler()
      train_features = scaler.fit_transform(train_features)
      val_features = scaler.transform(val_features)

      results = random_forest(train_features, train_dx, val_features, val_dx, n=n)
      print(f"Random Forest: {n} | Precision: {results[0]} | Recall: {results[1]} | F1: {results[2]} | PR-AUC: {results[3]}")
      csv_writer.writerow([n] + results)

Evaluate (n = 300)

In [ ]:
train_df = pd.read_csv(f'./abcde_features_train.csv')
val_df = pd.read_csv(f'./abcde_features_test.csv')

train_dx = train_df.iloc[:, 1].values
train_features = train_df.iloc[:, 3:].values
val_dx = val_df.iloc[:, 1].values
val_features = val_df.iloc[:, 2:].values

n = 300

# Scale the features
scaler = StandardScaler()
train_features = scaler.fit_transform(train_features)
val_features = scaler.transform(val_features)

results = random_forest(train_features, train_dx, val_features, val_dx, n=n)
print(f"Random Forest: {n} | Precision: {results[0]} | Recall: {results[1]} | F1: {results[2]} | PR-AUC: {results[3]}")

### SVM

In [ ]:
from sklearn.svm import SVC

def svm(x_train, y_train, x_val, y_val, kernel, degree=3):
    svm = SVC(kernel=kernel, degree=degree)  # 'linear', 'rbf', 'poly', 'sigmoid'
    svm.fit(x_train, y_train)
    y_pred = svm.predict(x_val)

    results = get_metrics(y_val, y_pred)
    return results

Fine Tune

In [ ]:
# Import PCA
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
import pandas as pd
import csv

train_df = pd.read_csv(f'./abcde_features_train.csv')

dx = train_df.iloc[:, 1].values
patient_ids = train_df.iloc[:, 2].values
features = train_df.iloc[:, 3:].values

with open(f'./isic2018_svm_fine_tune.csv', 'w', newline='') as csvfile:
  csv_writer = csv.writer(csvfile)
  csv_writer.writerow(["Number Learners", "Precision", "Recall", "F1", "PR-AUC"])

  for kernel in ["linear", "rbf", "sigmoid"]:
    for fold, (train_idx, val_idx) in enumerate(sgkf.split(features, dx, patient_ids)):
      train_features, val_features = features[train_idx], features[val_idx]
      train_dx, val_dx = dx[train_idx], dx[val_idx]

      # Scale the features
      scaler = StandardScaler()
      train_features = scaler.fit_transform(train_features)
      val_features = scaler.transform(val_features)

      results = svm(train_features, train_dx, val_features, val_dx, kernel=kernel)
      print(f"SVM: {kernel} | Precision: {results[0]} | Recall: {results[1]} | F1: {results[2]} | PR-AUC: {results[3]}")
      csv_writer.writerow([kernel] + results)

SVM with poly kernel

In [ ]:
train_df = pd.read_csv(f'./abcde_features_train.csv')

dx = train_df.iloc[:, 1].values
patient_ids = train_df.iloc[:, 2].values
features = train_df.iloc[:, 3:].values

with open(f'./isic2018_svm_fine_tune.csv', 'a', newline='') as csvfile:
  csv_writer = csv.writer(csvfile)

  for degree in [2, 3, 4, 5]:
    for fold, (train_idx, val_idx) in enumerate(sgkf.split(features, dx, patient_ids)):
      train_features, val_features = features[train_idx], features[val_idx]
      train_dx, val_dx = dx[train_idx], dx[val_idx]

      # Scale the features
      scaler = StandardScaler()
      train_features = scaler.fit_transform(train_features)
      val_features = scaler.transform(val_features)

      results = svm(train_features, train_dx, val_features, val_dx, kernel='poly', degree=degree)
      print(f"SVM: poly-{degree} | Precision: {results[0]} | Recall: {results[1]} | F1: {results[2]} | PR-AUC: {results[3]}")
      csv_writer.writerow([kernel] + results)

Evaluate (kernel = rbf)

In [ ]:
# Import PCA
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
import pandas as pd
import csv

train_df = pd.read_csv(f'./abcde_features_train.csv')
val_df = pd.read_csv(f'./abcde_features_test.csv')

train_dx = train_df.iloc[:, 1].values
train_features = train_df.iloc[:, 3:].values
val_dx = val_df.iloc[:, 1].values
val_features = val_df.iloc[:, 2:].values

kernel = "rbf"

# Scale the features
scaler = StandardScaler()
train_features = scaler.fit_transform(train_features)
val_features = scaler.transform(val_features)

results = svm(train_features, train_dx, val_features, val_dx, kernel=kernel)
print(f"SVM: {kernel} | Precision: {results[0]} | Recall: {results[1]} | F1: {results[2]} | PR-AUC: {results[3]}")